<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 El Examen Sorpresa: ¿Memorizaste o Entendiste? 📝
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/06%20-%20Modelamiento%20Estadistico/Para%20Dummies/02_Evaluacion_Modelo_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 02 del Módulo 06 (Evaluación del modelo)](../02_Evaluacion_del_Modelo_R2_MSE_y_Validacion_Cruzada.ipynb). Imagina un estudiante que se **aprende de memoria** las respuestas del simulacro: saca 5.0 en el simulacro y se hunde en el examen real, donde las preguntas cambian un poco. Un modelo puede hacer lo mismo: **memorizar** los datos con los que se entrenó y fallar con datos nuevos. Para saber si **entendió o memorizó**, hay que examinarlo con preguntas que **no vio**.

| En el colegio | En estadística |
|---|---|
| Las preguntas del simulacro (que el alumno estudió) | Los datos de **entrenamiento** |
| El examen sorpresa (preguntas nuevas) | Los datos de **prueba** |
| Memorizar las respuestas | **Sobreajuste** |
| Entender el tema | Un modelo que **generaliza** |
| Cinco exámenes distintos con las mismas preguntas rotadas | **Validación cruzada** |
| «El error típico de tu nota es de 0.5 puntos» | El **RMSE** (error típico en las mismas unidades) |
| «Explicaste el 80 % de por qué unos alumnos sacan más que otros» | El **R²** |

Al terminar podrás explicar:
1. Por qué un modelo muy flexible tiene **error pequeño en el simulacro** y **grande en el examen**.
2. Cómo la **validación cruzada** (varios exámenes) ayuda a escoger el modelo sin hacer trampa.
3. Por qué **agregar variables inútiles sube el R²** del entrenamiento aunque no sirvan.

> 🔗 Si luego quieres ver las versiones **con fórmulas, AIC/BIC, validación por grupos y por tiempo**: [cuaderno estándar](../02_Evaluacion_del_Modelo_R2_MSE_y_Validacion_Cruzada.ipynb). Taller del módulo: [`../../homeworks/Para Dummies/06_Modelamiento_Estadistico_Hands_On_Dummies.ipynb`](../../homeworks/Para%20Dummies/06_Modelamiento_Estadistico_Hands_On_Dummies.ipynb).

---
## 1. Simulacro Perfecto, Examen Malo 🎓

Medimos la **altura de 30 plantas** a lo largo de las semanas (datos **inventados**: crecen con una curva suave más un poco de azar). Ajustamos tres «estudiantes»: uno muy simple (una recta), uno razonable (una curva suave) y uno **memorioso** (una curva con 12 «vaivenes» que pasa casi por cada punto). Los examinamos en el **simulacro** (las 30 plantas de siempre) y en el **examen sorpresa** (30 plantas nuevas de la misma población). El error se mide con el **RMSE**: el error típico, en las mismas unidades que lo medido (cm).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
def plantas(n):
    semana = np.sort(rng.uniform(-1, 1, n))                                        # semana (reescalada)
    return semana, 20 + 8 * semana - 6 * semana ** 2 + 5 * semana ** 3 + rng.normal(0, 2, n)     # curva suave + azar (INVENTADO)

x_simulacro, y_simulacro = plantas(30)
x_examen, y_examen = plantas(30)

def rmse(y, pred):
    return np.sqrt(np.mean((y - pred) ** 2))

filas = {}
for nombre, grado in [("recta (muy simple)", 1), ("curva suave (grado 3)", 3), ("memorioso (grado 12)", 12)]:
    coef = np.polyfit(x_simulacro, y_simulacro, grado)
    filas[nombre] = {"error en el simulacro": rmse(y_simulacro, np.polyval(coef, x_simulacro)),
                     "error en el examen": rmse(y_examen, np.polyval(coef, x_examen))}
tabla = pd.DataFrame(filas).T
print(tabla.round(2).to_string())
assert tabla.loc["memorioso (grado 12)", "error en el simulacro"] < tabla.loc["curva suave (grado 3)", "error en el simulacro"]
assert tabla.loc["memorioso (grado 12)", "error en el examen"] > 1.5 * tabla.loc["curva suave (grado 3)", "error en el examen"]

malla = np.linspace(-1, 1, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(x_simulacro, y_simulacro, color="#f59e0b", label="simulacro (entrenamiento)", zorder=3)
ax.scatter(x_examen, y_examen, color="#64748b", marker="x", label="examen (datos nuevos)", zorder=3)
for grado, col in [(3, "#16a34a"), (12, "#dc2626")]:
    ax.plot(malla, np.polyval(np.polyfit(x_simulacro, y_simulacro, grado), malla), color=col, label=f"grado {grado}")
ax.set_ylim(0, 40); ax.set(xlabel="semana (reescalada)", ylabel="altura (cm)", title="El memorioso se pega a cada punto y se descontrola"); ax.legend(fontsize=8)
plt.show()

---
**Lección:** cuanto más flexible es el modelo, **menor** es su error en el simulacro (el memorioso falla 1.30 cm; la curva suave, 1.76), pero en el **examen** el memorioso falla **52.9 cm** y la curva suave solo 1.68 cm. Si solo miraras el simulacro, escogerías al peor estudiante. El error en datos que el modelo **no vio** es el que cuenta.

---
## 2. Cinco Exámenes en Lugar de Uno: Validación Cruzada 🔁

Casi nunca tenemos plantas «de repuesto» para un examen sorpresa. Truco: partimos los datos en **5 grupos** (como 5 mazos de cartas). Cada vez **guardamos un mazo** para el examen, entrenamos con los otros 4 y medimos el error; repetimos 5 veces rotando el mazo. El promedio de los 5 errores es el **error de examen** estimado, sin gastar datos nuevos. Con eso comparamos del grado 1 al 12 y escogemos el mejor.

In [ ]:
def error_cinco_examenes(x, y, grado, semilla=0):
    '''Partimos en 5 mazos; cada mazo hace de examen una vez. Devolvemos el RMSE promedio de los 5 examenes.'''
    mazos = np.array_split(np.random.default_rng(semilla).permutation(len(x)), 5)
    errores = []
    for examen in mazos:
        estudio = np.setdiff1d(np.arange(len(x)), examen)
        coef = np.polyfit(x[estudio], y[estudio], grado)
        errores.append(rmse(y[examen], np.polyval(coef, x[examen])))
    return np.mean(errores)

grados = range(1, 13)
errores_cv = pd.Series({g: error_cinco_examenes(x_simulacro, y_simulacro, g) for g in grados})
mejor = errores_cv.idxmin()
print(errores_cv.round(2).to_string())
print(f"\nMenor error de examen: grado {mejor} (la verdad inventada era una curva de grado 3)")
assert 2 <= mejor <= 5 and errores_cv[12] > errores_cv[mejor]

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(errores_cv.index, errores_cv.values, marker="o", color="#f59e0b")
ax.axvline(mejor, color="#16a34a", ls="--", label=f"mejor: grado {mejor}")
ax.set(xlabel="flexibilidad del modelo (grado)", ylabel="error de examen (cm)", title="Pocos vaivenes no alcanzan; demasiados, memorizan"); ax.legend()
plt.show()

---
**Lección:** el mejor es el grado 4 (error de examen ≈ 2.0 cm), con el grado 3 —el que usamos para inventar los datos— muy cerca (≈ 2.1); la recta se queda corta (2.35) y del grado 9 en adelante el error se dispara (34.7 a 247 cm). Ojo: con solo 30 plantas el «mejor» puede moverse un grado de una muestra a otra; lo importante es la **forma** de la curva (baja, toca fondo y se dispara).

---
## 3. Variables Inútiles: el R² Que Miente 🎭

El **R²** dice «qué fracción de las diferencias entre plantas explica el modelo» (0 = nada, 1 = todo). Pero hay una trampa: si le **agregas columnas de puro azar** (números de lotería sin relación con nada), el R² en el simulacro **sube igual**, porque el modelo las usa para memorizar un poquito más. En el examen, en cambio, esas columnas estorban.

In [ ]:
def r2(y, pred):
    return 1 - np.sum((y - pred) ** 2) / np.sum((y - y.mean()) ** 2)

rng = np.random.default_rng(5)
n = 60
util = rng.normal(size=(2 * n, 1))
y_total = 10 + 3 * util[:, 0] + rng.normal(0, 3, 2 * n)                             # solo UNA columna sirve (INVENTADO)
res = []
for k_basura in (0, 10, 25, 40):
    X = np.column_stack([np.ones(2 * n), util, rng.normal(size=(2 * n, k_basura))])     # k_basura columnas de loteria
    b = np.linalg.lstsq(X[:n], y_total[:n], rcond=None)[0]
    res.append({"columnas basura": k_basura, "R2 en el simulacro": r2(y_total[:n], X[:n] @ b), "R2 en el examen": r2(y_total[n:], X[n:] @ b)})
res = pd.DataFrame(res).set_index("columnas basura")
print(res.round(2).to_string())
assert res["R2 en el simulacro"].iloc[-1] > res["R2 en el simulacro"].iloc[0] + 0.2 and res["R2 en el examen"].iloc[-1] < res["R2 en el examen"].iloc[0]

---
##### 🛠️ Práctica 1: Tu turno: ¿qué grado elige la validación cruzada con otros datos?

Con la función `error_cinco_examenes` de la sección 2:

1. Genera 40 plantas nuevas con `x_n, y_n = plantas(40)`.
2. Calcula `errores_n`, el error de los cinco exámenes para cada grado de 1 a 8 con `error_cinco_examenes(x_n, y_n, g)`.
3. Guarda el mejor en `mejor_n = ...` y verifica con `assert` que `mejor_n >= 2` y que el error del grado 1 (recta) es **mayor** que el del mejor. ¿Coincide con la curva que usamos para inventar los datos?

In [ ]:
# Escribe tu código aquí

# x_n, y_n = plantas(40)
# errores_n = ...
# mejor_n = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
x_n, y_n = plantas(40)
errores_n = pd.Series({g: error_cinco_examenes(x_n, y_n, g) for g in range(1, 9)})
mejor_n = errores_n.idxmin()
print(errores_n.round(2).to_dict(), "-> mejor grado:", mejor_n)

assert mejor_n >= 2 and errores_n[1] > errores_n[mejor_n]
```
</details>

---


---
**Lección:** al agregar columnas de lotería, el R² del simulacro **sube** de 0.43 a 0.84 (parece que el modelo «explica más») mientras el del examen **cae** de 0.33 a −1.27, es decir, se vuelve negativo (peor que adivinar el promedio). Por eso **nunca** se celebra el R² del entrenamiento: se mira el de datos nuevos o el de la validación cruzada.

---
## Resumen en una frase 🎯

> **Un modelo se evalúa con preguntas que no vio (datos nuevos o validación cruzada), porque el error en el simulacro premia al que memoriza y el R² de entrenamiento sube aunque agregues columnas inútiles.**

### Lo que aprendiste hoy:
- ✅ Un modelo **demasiado flexible** memoriza: error mínimo en el simulacro, error alto en el examen (**sobreajuste**).
- ✅ La **validación cruzada** reparte los datos en mazos y rota el examen, para estimar el error sin gastar datos nuevos.
- ✅ El **RMSE** es el error típico en las mismas unidades (cm); el **R²** es la fracción de diferencias que se explica.
- ✅ Agregar **variables inútiles** sube el R² de entrenamiento pero **empeora** el examen.
- ✅ Se escoge el modelo con el **menor error de examen**, no con el menor error de simulacro.

---
### 🧠 ¿Ya lo tienes claro?

1. Un modelo tiene R² = 0.98 con los datos que usó para aprender y R² = 0.10 con datos nuevos. ¿Qué le pasó? ¿Qué le recomendarías?
2. ¿Por qué la validación cruzada usa cada dato una vez como «examen» y no varias veces como entrenamiento y examen al mismo tiempo?
3. Un vendedor te muestra un modelo de precios que «explica el 99 %». ¿Qué preguntas le harías antes de creerle?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>